# Lab 2: 语音识别 — Qwen3-ASR + OpenVINO

Qwen3-ASR 是通义千问团队推出的自动语音识别模型，支持 52 种语言和方言的语音识别和语言识别。该模型基于 Qwen3-Omni 的强大音频理解能力和大规模语音训练数据，在开源 ASR 模型中达到了最先进的性能。

**主要特点:**
- 🌍 支持 30 种语言和 22 种中文方言的语音识别
- ⚡ 0.6B 版本在保持高精度的同时实现极高吞吐量
- 🎤 支持流式/离线统一推理
- 📏 支持长音频转写

在本实验中，我们将：
1. 从 ModelScope 下载预转换的 OpenVINO 模型
2. 使用 OpenVINO 加载并运行语音识别
3. 体验交互式 Gradio 语音识别演示

#### 目录：
- [下载模型](#下载模型)
- [选择推理设备](#选择推理设备)
- [加载模型](#加载模型)
- [运行语音识别](#运行语音识别)
- [交互式演示](#交互式演示)

## 环境准备
[返回目录 ⬆️](#目录：)

### Step 1：创建 Python 虚拟环境

```bash
python -m venv .venv
```

激活虚拟环境：
```bash
.venv/Scripts/activate
```

### Step 2：安装依赖

> ⚠️ 注意：release 版本的 ov-ASR 有问题，请使用 nightly 版本的 OpenVINO。

```bash
pip install --upgrade pip
pip install modelscope soundfile
pip install openvino~=2026.2.0.0.dev openvino_genai~=2026.2.0.0.dev openvino-tokenizers~=2026.2.0.0.dev --extra-index-url https://storage.openvinotoolkit.org/simple/wheels/nightly
```

## 下载模型
[返回目录 ⬆️](#目录：)

从 ModelScope 下载已转换的 Qwen3-ASR-0.6B OpenVINO 模型。如果模型已存在则跳过下载。

In [ ]:
from pathlib import Path

model_dir = Path("Qwen3-ASR-0.6B-fp16-ov")

if not model_dir.exists():
    from modelscope import snapshot_download
    snapshot_download("snake7gun/Qwen3-ASR-0.6B-fp16-ov", local_dir=str(model_dir))
    print(f"模型已下载到: {model_dir}")
else:
    print(f"模型已存在: {model_dir}，跳过下载")

## 选择推理设备
[返回目录 ⬆️](#目录：)

选择用于推理的设备。CPU 兼容性最好，GPU 可在支持的硬件上提供加速。

In [ ]:
from notebook_utils import device_widget

device = device_widget("GPU", exclude=["NPU"])
device

## 加载模型
[返回目录 ⬆️](#目录：)

使用 `OVQwen3ASRModel` 加载 OpenVINO 模型。该 wrapper 提供与原始 `Qwen3ASRModel` 相同的 API 接口。

In [ ]:
from qwen_3_asr_helper import OVQwen3ASRModel

ov_model = OVQwen3ASRModel.from_pretrained(
    model_dir=str(model_dir),
    device=device.value,
    max_inference_batch_size=32,
    max_new_tokens=256,
)

print(f"支持的语言: {ov_model.get_supported_languages()}")

## 运行语音识别
[返回目录 ⬆️](#目录：)

让我们用一段英文音频来测试模型。模型支持多种音频格式（wav, mp3, flac），也可以接受文件路径、URL 或 NumPy 数组。

In [ ]:
import urllib.request

# 下载测试音频
sample_audio_en = Path("sample_en.wav")

if not sample_audio_en.exists():
    print("正在下载英文示例音频...")
    urllib.request.urlretrieve("https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen3-ASR-Repo/asr_en.wav", sample_audio_en)
    print(f"已下载: {sample_audio_en}")

# 运行转写
results = ov_model.transcribe(
    audio=str(sample_audio_en),
    language=None,  # 自动检测语言
)

print(f"\n检测到的语言: {results[0].language}")
print(f"转写结果: {results[0].text}")